In [2]:
import pandas as pd
import numpy as np
import pickle

# Scikit-Learn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Matplotlib (für Visualisierung, falls benötigt)
import matplotlib.pyplot as plt

# Keras/Tensorflow für das neuronale Netz
from scikeras.wrappers import KerasRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

import pandas as pd

df = pd.read_csv("../data/transform/pro/stats_with_rating.csv")

# Information
This notebook defines the necessary preprocessing steps to prepare the player_stats dataset for model training. A subset of these steps will later be exported into a standalone Python module and integrated into the data loading and preprocessing pipeline used for inference. These reusable steps are explicitly labeled throughout the notebook.

The remaining steps are specific to the data integration process and are required to merge data from Transfermarkt (TM) with data from Sofascore. This integration is necessary to link player_stats with the corresponding player ratings (grades) and is not part of the final preprocessing pipeline used during model deployment.

In [3]:
X = df.drop(columns=['rating'])
y = df['rating']

# Split randomly
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Split nach spielern
unique_players = df['player_id'].unique()

train_players, test_players = train_test_split(
    unique_players,
    test_size=0.2,
    random_state=42
)

mask_train_player = df['player_id'].isin(train_players)
mask_test_player = df['player_id'].isin(test_players)

X_train_player = X[mask_train_player]
X_test_player = X[mask_test_player]

y_train_player = y[mask_train_player]
y_test_player = y[mask_test_player]


# Split nach spielen
unique_matches = df['match_id'].unique()

train_matches, test_matches = train_test_split(
    unique_matches,
    test_size=0.2,
    random_state=42
)

mask_train_matches = df['match_id'].isin(train_matches)
mask_test_matches = df['match_id'].isin(test_matches)

X_train_matches = X[mask_train_matches]
X_test_matches = X[mask_test_matches]

y_train_matches = y[mask_train_matches]
y_test_matches = y[mask_test_matches]

In [4]:
cols = ["player_id", "match_id", "club_id", "player_name", "date",  "home_club_id", "away_club_id", "home_goals", "away_goals"]
df = df.drop(columns = cols, errors="ignore")

num_cols = ["goals", "assists", "minutes", "team_goals", "team_conceded"]
cat_cols = ["position", "result"]
bool_cols = ["yellow", "red", "start_eleven"]

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("bool", "passthrough", bool_cols),
    ]
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.3180702544486868
RMSE: 0.42906391984486403


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("bool", "passthrough", bool_cols),
    ]
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_grid = {
    "model__n_estimators": [100, 400, 1000],
    "model__max_depth": [10, 20, 30],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)


best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("Best Params:", grid.best_params_)
print("MAE:", mae)
print("RMSE:", rmse)


Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Params: {'model__max_depth': 30, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 1000}
MAE: 0.18223098147452454
RMSE: 0.30578231971745734


In [7]:
with open("best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("bool", "passthrough", bool_cols),
    ]
)

# Ridge
pipeline_ridge = Pipeline([
    ("preprocessing", preprocessor),
    ("model", Ridge(alpha=1.0))
])

pipeline_ridge.fit(X_train, y_train)
y_pred_ridge = pipeline_ridge.predict(X_test)

mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = root_mean_squared_error(y_test, y_pred_ridge)

print("Ridge MAE:", mae_ridge)
print("Ridge RMSE:", rmse_ridge)

# Lasso
pipeline_lasso = Pipeline([
    ("preprocessing", preprocessor),
    ("model", Lasso(alpha=0.1))
])

pipeline_lasso.fit(X_train, y_train)
y_pred_lasso = pipeline_lasso.predict(X_test)

mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
rmse_lasso = root_mean_squared_error(y_test, y_pred_lasso)

print("Lasso MAE:", mae_lasso)
print("Lasso RMSE:", rmse_lasso)

Ridge MAE: 0.31806009708926336
Ridge RMSE: 0.42905458634268107
Lasso MAE: 0.36618784157128004
Lasso RMSE: 0.4956819817314097


In [9]:
from scikeras.wrappers import KerasRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

def build_model(meta, activation="relu", alpha=0.001, dropout_rate=0.2, momentum=0.0):
    model = Sequential()

    # Input automatisch von scikeras
    model.add(Dense(128, activation=activation, input_shape=(meta["n_features_in_"],)))
    model.add(Dropout(dropout_rate))
    
    model.add(Dense(64, activation=activation))
    model.add(Dropout(dropout_rate))
    
    model.add(Dense(32, activation=activation))

    model.add(Dense(1))

    optimizer = SGD(learning_rate=alpha, momentum=momentum)

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

nn = KerasRegressor(
    model=build_model,
    epochs=50,
    batch_size=32,
    verbose=0
)

pipeline_nn = Pipeline([
    ("preprocessing", preprocessor),
    ("model", nn)
])

param_grid = {
    "model__model__activation": ["relu", "tanh"],
    "model__model__alpha": [0.001, 0.01],
    "model__model__dropout_rate": [0.2, 0.3],
    "model__model__momentum": [0.0, 0.9]
}

grid = GridSearchCV(
    pipeline_nn,
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

y_pred_nn = grid.predict(X_test)

mae_nn = mean_absolute_error(y_test, y_pred_nn)
rmse_nn = root_mean_squared_error(y_test, y_pred_nn)

print("NN MAE:", mae_nn)
print("NN RMSE:", rmse_nn)

KeyboardInterrupt: 

## Split-Performance
In dieser Zelle wird geschaut, ob für den Random Forest die performance deutlich schlechter wird, wenn wir nacht "logik" Splitten.

In [11]:
best_params = {
    "n_estimators": 1000,
    "max_depth": 30,
    "min_samples_split": 2,
    "min_samples_leaf": 1
}

best_model_mse = 0.18223098147452454
best_model_rmse = 0.30578231971745734

model_player = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1, **best_params))
])

model_matches = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1, **best_params))
])

model_player.fit(X_train_player, y_train_player)
model_matches.fit(X_train_matches, y_train_matches)

y_pred_player = model_player.predict(X_test_player)
y_pred_matches = model_matches.predict(X_test_matches)

rmse_player = root_mean_squared_error(y_test_player, y_pred_player)
rmse_matches = root_mean_squared_error(y_test_matches, y_pred_matches)

print("MSE Player Split:", rmse_player**2)
print("RMSE Player Split:", rmse_player)

print("MSE Match Split:", rmse_matches**2)
print("RMSE Match Split:", rmse_matches)

print("Difference Player MSE:", abs(best_model_mse - rmse_player**2))
print("Difference Player RMSE:", abs(best_model_rmse - rmse_player))

print("Difference Match MSE:", abs(best_model_mse - rmse_matches**2))
print("Difference Match RMSE:", abs(best_model_rmse - rmse_matches))

MSE Player Split: 0.22840523298202836
RMSE Player Split: 0.4779176006196344
MSE Match Split: 0.22157828657908915
RMSE Match Split: 0.47072102840120617
Difference Player MSE: 0.046174251507503816
Difference Player RMSE: 0.17213528090217706
Difference Match MSE: 0.039347305104564606
Difference Match RMSE: 0.16493870868374882


# Auswahl Modell
Wir entschieden uns für den RF der auf dem Matches-Split trainiert wird, da die Daten dort abhängig voneinander sind und wir so Data-Leakage vermeiden.

In [12]:
# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("bool", "passthrough", bool_cols),
    ]
)

# Finales Modell (basierend auf Match Split Best Params)
final_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        max_depth=20,
        min_samples_leaf=1,
        min_samples_split=2,
        n_estimators=1000
    ))
])

# Trainieren auf ALLEN Daten
final_model.fit(X, y)

# Speichern
with open("best_model.pkl", "wb") as f:
    pickle.dump(final_model, f)